Load demo data to config table

In [ ]:
dbutils.widgets.text("catalog_name", "", "Catalog name")
dbutils.widgets.text("schema_name", "", "Schema name")

In [ ]:
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")

In [ ]:
query = f"""
USE CATALOG {catalog_name};
"""

spark.sql(query)

query = f"""
USE SCHEMA {schema_name};
"""

spark.sql(query)

Insert some demo metadata config

In [ ]:
query = """
INSERT INTO etl_meta_config (emc_task_id, emc_seq, emc_target_layer, emc_func_type, emc_func_dtls, emc_target, emc_out_mode, emc_table_retention, emc_target_storage_loc, emc_active, emc_desc)
VALUES 
  ("AUTOLOAD1", 
  1, 
  UPPER("Bronze"), 
  UPPER("Notebook"),
  PARSE_JSON('{"notebook_path":"../demo/autoloader_example", "parms":{"format":"CSV","header": "true", "inferColumnTypes": "false", "schemaLocation":"/Volumes/dev_etl/volumes/etl_autoloader_folder/raw/schema/autoload.json","source_path":"/Volumes/dev_etl/volumes/etl_autoloader_folder/raw/data/","outputMode":"append","checkpoint":"/Volumes/dev_etl/volumes/etl_autoloader_folder/raw/checkpoint", "mergeSchema":"true","emc_target":"dev_etl.bronze.test1"}, "timeout":120}'),
  "dev_etl.bronze.test1",
  UPPER("append"), 
  12, 
  "", 
  "Y", 
  "Test bronze load using autoloader"),
  ("SILVERLOAD1", 
  1, 
  UPPER("Silver"), 
  UPPER("sqlload"),
  PARSE_JSON('{"sql_path":"../demo/sqlload.txt"}'),
  "", 
  UPPER("append"), 
  12,
  "", 
  "Y", 
  "Load SQL statements"),
  ("SILVERLOAD1", 
  2, 
  UPPER("Silver"), 
  UPPER("sql"),
  PARSE_JSON('{"sql_var":"silverload", "sql_parms":""}'),
  "dev_etl.silver.test", 
  UPPER("append"), 
  12,
  "", 
  "Y", 
  "Load Load SQL statements"),
  ("GOLDLOAD1", 
  1, 
  UPPER("Gold"), 
  UPPER("sqlload"),
  PARSE_JSON('{"sql_path":"../demo/sqlload.txt"}'),
  "", 
  UPPER("append"), 
  12,
  "", 
  "Y", 
  "Load SQL statements"),
  ("GOLDLOAD1", 
  2, 
  UPPER("Gold"), 
  UPPER("sql"),
  PARSE_JSON('{"sql_var":"goldload", "sql_parms":""}'),
  "dev_etl.gold.test", 
  UPPER("overwrite"), 
  12,
  "", 
  "Y", 
  "Load Gold tables"),
  ("NOTEBOOK2", 
  1, 
  UPPER("Gold"), 
  UPPER("Notebook"),
  PARSE_JSON('{"notebook_path":"../demo/demo_notebook", "parms": {"key1":"value1","key2":"value2"}, "timeout":240}'),
  "dev_etl.gold.agg_age", 
  UPPER("overwrite"), 
  12,
  "", 
  "Y", 
  "Load Gold tables")
"""

spark.sql(query)

Create the demo catalog and schemas

In [ ]:
# query = """
# CREATE CATALOG IF NOT EXISTS dev_etl;
# """
# spark.sql(query)

query = f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.bronze;
"""
spark.sql(query)

query = f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.volumes;
"""
spark.sql(query)

query = f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.silver;
"""
spark.sql(query)

query = f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.gold;
"""
spark.sql(query)

Create the demo tables and insert dummy data

In [ ]:
query = f"""
CREATE OR REPLACE TABLE {catalog_name}.bronze.test
(test_id STRING NOT NULL COMMENT "identifier",
test_name STRING COMMENT "Name",
test_age INT COMMENT "Age",
test_country STRING COMMENT "Country code",
test_email STRING COMMENT "Email",
test_load_date DATE COMMENT "Load date")
COMMENT "Bronze dummy data table" 
"""
spark.sql(query)


query = f"""
INSERT INTO {catalog_name}.bronze.test 
VALUES 
("0000001", "Name1", 15, "UK", "Name1@gmail.com", "2022-10-20"),
("0000002", "Name2", 25, "US", "Name2@gmail.com", "2022-10-20"),
("0000003", "Name3", 35, "UK", "Name3@gmail.com", "2022-10-20"),
("0000004", "Name4", 45, "NZ", "Name4@gmail.com", "2022-10-20"),
("0000005", "Name5", 44, "", "Name5@gmail.com", "2022-10-20"),
("0000006", "Name6", 15, "UK", "Name6@gmail.com", "2022-10-21"),
("0000007", "Name7", 25, "US", "Name7gmail.com", "2022-10-21"),
("0000008", "Name8", 35, "UK", "Name8@gmail.com", "2022-10-21");
"""
spark.sql(query)

query = f"""
CREATE OR REPLACE TABLE {catalog_name}.silver.test
(dst_id STRING NOT NULL COMMENT "identifier",
dst_name STRING COMMENT "Name",
dst_age INT COMMENT "Age",
dst_country STRING COMMENT "Country",
dst_email STRING COMMENT "Email",
dst_load_date DATE COMMENT "Load date")
COMMENT "Silver dummy data table"
"""
spark.sql(query)

query = f"""
CREATE OR REPLACE TABLE {catalog_name}.gold.test
(dgt_country STRING NOT NULL COMMENT "Country",
dgt_count STRING COMMENT "Number")
COMMENT "Gold dummy data table"
"""
spark.sql(query)

query = f"""
CREATE OR REPLACE TABLE {catalog_name}.gold.agg_age
(dga_age_bracket STRING NOT NULL COMMENT "Age bracket",
dga_count_per_bracket INT COMMENT "Count")
COMMENT "Gold dummy data table"
"""
spark.sql(query)

query = f"""
CREATE OR REPLACE TABLE {catalog_name}.bronze.test1
(Name STRING COMMENT "Name",
Email STRING COMMENT "Email")
COMMENT "Bronze autoloader table"
"""
spark.sql(query)


# Create volume , directories and demo files

In [ ]:
query = f"""
CREATE VOLUME IF NOT EXISTS {catalog_name}.volumes.etl_autoloader_folder
COMMENT 'Volume for ETL Auto Loader demo'
"""

spark.sql(query)

In [ ]:
import os

# Path to your volume
volume_path = f"/Volumes/{catalog_name}/volumes/etl_autoloader_folder"

# Create a subdirectory
subdir_path = os.path.join(volume_path, "raw/data")
dbutils.fs.mkdirs(subdir_path)
print(f"Directory created at: {subdir_path}")

In [ ]:
import os

# 1. Define your CSV data in a variable
csv_data = """Name,Email,Age,Countrycode
Alice,Alice@gmail.com,30,US
Bob,Bob@test.com,25,ID
Charlie,CharlesXavier@hi.com,35,NZ
"""

# 2. Path to your Volume and subdirectory
volume_path = subdir_path

# 3. Write the CSV data to a file in the Volume
outfile = f"{volume_path}/sample.csv"
with open(outfile, "w") as f:
    f.write(csv_data)

print(f"CSV file successfully written to: {volume_path}/sample.csv")